> **Research record.** Cleaned copy of `800-research-users-locations/1000-define-geohashes-potential-third-places.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Selects geohashes with POIs that can host remote work and saves them as potential third places.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

## Save third places separately

In [ ]:
import sys

project_root = "../.."
if project_root not in sys.path:
    sys.path.append(project_root)

from geofunctions import utils

### V1: OSM data

In [ ]:
# Only geohashes with 3rd places (excluding coworkings)
# Updated 24.1.2026 : exclude geohashes with educational and medical institutions, also transport hubs
utils.read_query_to_dataframe(
    """create or replace table phd_datasets.geohash_thirdplaces_osm cluster by geohash as 
        SELECT
            geohash,
            cafe_count+g.coffee_shop_count as cafe_count,
            g.library_count,
            g.bookstore_count+ g.bookshop_count as bookstore_count,
            g.community_center_count+ g.community_centre_count as community_center_count,
            g.count_office_like_places,
            g.count_remote_possible_places,
        FROM phd_datasets.geohash_features g
        where (g.coffee_shop_count + g.cafe_count + g.library_count 
        +g.bookstore_count+ g.bookshop_count + g.community_center_count+ g.community_centre_count  > 0)
        and (count_universities+count_colleges+count_schools+count_medical_institutions+count_transport_hubs=0) and flag_work_zone=0 """
)

### V2: Google 3rd places

In [ ]:
"""create or replace table phd_datasets.geohash_thirdplaces_google cluster by geohash as 
        SELECT
            geohash,
            sum(CASE when primarytype in  (
                "cafe",
                "coffee_shop"
                ) then 1 else 0 end) as cafe_count,
             sum(CASE when primarytype  = "bakery" then 1 else 0 end) as bakery_count,
             sum(CASE when primarytype  = "tea_house" then 1 else 0 end) as tea_house_count,
             sum(CASE when primarytype  = "dessert_restaurant" then 1 else 0 end) as dessert_restaurant_count,
             sum(CASE when primarytype  = "cat_cafe" then 1 else 0 end) as cat_cafe_count,
             sum(CASE when primarytype  = "dog_cafe" then 1 else 0 end) as dog_cafe_count,
             sum(CASE when primarytype  = "breakfast_restaurant" then 1 else 0 end) as breakfast_restaurant_count,
             sum(CASE when primarytype  = "brunch_restaurant" then 1 else 0 end) as brunch_restaurant_count,
            sum(CASE when primarytype = "library" then 1 else 0 end) as library_count,
            sum(CASE when primarytype = "book_store" then 1 else 0 end) as bookstore_count,
            sum(CASE when primarytype = "museum" then 1 else 0 end) as museum_count,
            sum(CASE when primarytype = "cultural_center" then 1 else 0 end) as cultural_center_count,
            sum(CASE when primarytype = "community_center" then 1 else 0 end) as community_center_count
        FROM phd-habidatum.phd_adm.geohashes_with_sa_gushdan g
        JOIN phd-habidatum.phd_pois.google_3rdplaces_cities p
        on st_within(p.geometry, g.geometry)
        group by geohash """